---
title: ""
lang: uk
format:
  typst:
    mainfont: "Times New Roman"
    fontsize: 14pt
    margin: { x: 2.5cm, y: 2.5cm }
  docx: default
  pdf:
    pdf-engine: lualatex
    mainfont: "Times New Roman"
---

```{=typst}
// Configure the page.
#set page(
  paper: "a4",
  // The margins depend on the paper size.
  margin: (x: 2cm, y: 1.5cm),
  numbering:  (current, total) => if current > 1 { current }, number-align: right, 
)

// Configure styling
#set par(
  leading: 0.8em,
  spacing: 1em,
  justify: true,
)
#show table: set text(size: 9pt)

#align(top + center)[
  Міністерство освіти і науки України \
  Національний університет «Львівська політехніка» \
  Кафедра систем штучного інтелекту \
  \
  #image("logo.png", width: 6cm)
  \
  #set par(spacing: 0.4em)
  #text(size: 24pt, weight: "bold")[Звіт]
  #set par(spacing: 0.8em)

  *про виконання лабораторної роботи № 6* \
  _з дисципліни:_ «Обробка мови методами штучного інтелекту»
]

#align(bottom + right)[
  Виконав: \
  студент групи ШІ-42 \
  Горишевський Олексій Олександрович
]

#pagebreak()
```

# RNN Quest

1. Завантажте пакет із завданням за посиланням вище ^
2. Розпакуйте архів
3. Прочитайте README та виконайте завдання
  - Завдання не залежить від мови програмування
  - Завдання не залежить від фреймоворку. Pytorch, NumPy, Keras або взагалі без сторонніх бібліотек -- все можливо
4. Відправте відповідь нижче

In [1]:
%pip install --quiet --ignore-installed https://static.lp-nlp.com/pypi/lpnlp-2025.10.14-py3-none-any.whl requests==2.32.4 numpy

Note: you may need to restart the kernel to use updated packages.


In [2]:
import lpnlp

lab = lpnlp.start(
    email="oleksii.ivanov.shi.2023@lpnu.ua",
    lab="quest_rnn"
)

Ваше завдання: https://static.lp-nlp.com/quest-rnn/3f969ee462e6.zip. Удачі! █


## Constants

In [3]:
DIR = "lab-6-data/"
BOS_TOKEN, EOS_TOKEN = "[", "]"

# Elman RNN parameters
embedding_dim = 128
hidden_dim = 256
vocab_size = 133

## Downloading the data

In [4]:
import zipfile
import requests
from pathlib import Path

url = lab.payload["package_url"]

dir = Path(DIR)


def download_and_extract(url, dir):
    dir.mkdir(parents=True, exist_ok=True)

    # store the package
    response = requests.get(url)
    Path(DIR + "/package.zip").write_bytes(response.content)

    # extract the package
    with zipfile.ZipFile(DIR + "/package.zip", "r") as zip_ref:
        zip_ref.extractall(DIR)


if not dir.exists() or not any(dir.iterdir()):
    download_and_extract(url, dir)

## Importing the data

In [5]:
import json
import numpy as np


def load(name: str):
    with open(DIR + name + ".json", "r") as f:
        return np.array(json.load(f))


E = load("embedding.weight")

with open(DIR + "vocab.json", "r") as f:
    vocab_values: list[str] = json.load(f)
vocab: dict[str, int] = {v: i for i, v in enumerate(vocab_values)}
vocab_inv: dict[int, str] = dict(enumerate(vocab_values))

W_h = load("W_h.weight")
U_h = load("U_h.weight")

W_y = load("W_y.weight")
b = load("W_y.bias")

print("Shapes of loaded matrices:\n")
for name, M in [("E", E), ("W_h", W_h), ("U_h", U_h), ("W_y", W_y), ("b", b)]:
    print(name, M.shape)

Shapes of loaded matrices:

E (133, 128)
W_h (256, 128)
U_h (256, 256)
W_y (133, 256)
b (133,)


## Elman RNN

In [6]:
def generate_sequence(start_token: str, max_length: int) -> list[str]:
    if start_token not in vocab:
        raise ValueError(f"Start token '{start_token}' not in vocabulary.")

    x_t = E[vocab[start_token]]
    h_prev = np.zeros(hidden_dim)
    sequence: list[str] = [start_token]

    for _ in range(max_length):
        h_prev = np.tanh(W_h @ x_t + U_h @ h_prev)
        y_t = W_y @ h_prev + b

        # Greedy sampling
        # for stochastic sampling, use np.random.choice with p_t
        next_token_idx = int(np.argmax(y_t))
        next_token = vocab_inv[next_token_idx]

        if next_token == EOS_TOKEN:
            break

        sequence.append(next_token)

        x_t = E[next_token_idx]

    return sequence


def predict(start: str = BOS_TOKEN) -> str:
    sequence = generate_sequence(start, max_length=2**8)
    sequence_without_bos = sequence[1:] if sequence[0] == BOS_TOKEN else sequence
    return "".join(sequence_without_bos).replace(EOS_TOKEN, "")

## Generating an answer

In [7]:
full_answer = predict()
print("Full answer:", full_answer, sep="\n", end="\n\n")

url = predict("h")
print("URL:", url, sep="\n")

Full answer:
Назва статті: https://arxiv.org/abs/2303.12712

URL:
https://arxiv.org/abs/2303.12712


In [8]:
lab.answer("Sparks of Artificial General Intelligence: Early experiments with GPT-4")

Відповідь правильна ✅
You did it! 🚀 Next step: https://tally.so/r/wMXM0p
